# 02 · Validación (tarea final del job)
1. Muestra los conteos de cada tabla de Bronze, Silver y Gold.
2. Muestra los registros rechazados por fuente (silver.cuarentena).
3. **Falla** si alguna tabla de Gold queda vacía.

In [ ]:
dbutils.widgets.text("catalog", "electrocasa_dev", "Catálogo del proyecto")
catalogo = dbutils.widgets.get("catalog")
print("Catálogo:", catalogo)

In [ ]:
from pyspark.sql import functions as F

def tablas(schema):
    return [r.tableName for r in spark.sql(f"SHOW TABLES IN {catalogo}.{schema}").collect()
            if not r.isTemporary]

conteos = []
for schema in ["bronze", "silver", "gold"]:
    for t in sorted(tablas(schema)):
        n = spark.table(f"{catalogo}.{schema}.{t}").count()
        conteos.append((schema, t, n))
df_conteos = spark.createDataFrame(conteos, "capa string, tabla string, filas long")
display(df_conteos)

In [ ]:
rechazos = (
    spark.table(f"{catalogo}.silver.cuarentena")
    .groupBy("fuente", "politica")
    .agg(F.count("*").alias("registros"))
    .orderBy("fuente", "politica")
)
display(rechazos)

detalle = (
    spark.table(f"{catalogo}.silver.cuarentena")
    .select("fuente", "politica", F.explode("reglas_fallidas").alias("regla"))
    .groupBy("fuente", "politica", "regla").count()
    .orderBy("fuente", F.desc("count"))
)
display(detalle)

In [ ]:
# Tabla en Markdown para pegar en el README después de cada lote
filas = df_conteos.collect()
print("| Capa | Tabla | Filas |\n|---|---|---:|")
for r in filas:
    print(f"| {r.capa} | {r.tabla} | {r.filas:,} |")
print()
print("| Fuente | Política | Registros |\n|---|---|---:|")
for r in rechazos.collect():
    print(f"| {r.fuente} | {r.politica} | {r.registros:,} |")

In [ ]:
vacias = [r.tabla for r in filas if r.capa == "gold" and r.filas == 0]
esperadas = {"ventas_netas_sucursal_mes", "top_productos_unidades", "top_productos_tasa_devolucion",
             "dotacion_sucursal_mes", "ventas_vendedor_sucursal_mes",
             "resenas_negativas_categoria_mes", "despachos_courier", "empleados_vigentes"}
faltantes = esperadas - {r.tabla for r in filas if r.capa == "gold"}
if vacias or faltantes:
    raise Exception(f"Validación fallida. Gold vacías: {vacias}. Gold faltantes: {sorted(faltantes)}")
print("Validación OK: todas las tablas de Gold tienen filas.")